In [ ]:
from pathlib import Path

import numpy as np
from gwosc.datasets import event_gps
from gwpy.timeseries import TimeSeries

In [ ]:
EVENTO = "GW150914"
MEDIA_VENTANA = 16  # segundos a cada lado del evento
DATOS = Path("../datos")
DURACION_FFT = 4  # segundos por segmento al estimar el espectro
BANDA = (35, 350)  # Hz
RECORTE = (-0.2, 0.1)  # segundos respecto a t0
RETRASO_MAXIMO = 0.010  # 3000 km entre H1 y L1 a la velocidad de la luz
DATOS.mkdir(exist_ok=True)


def gps_evento(evento):
    # GWOSC se consulta una sola vez; después basta la copia de datos/, también sin red
    archivo = DATOS / f"{evento}_gps.txt"
    if archivo.exists():
        return float(archivo.read_text())
    t0 = event_gps(evento)
    archivo.write_text(repr(t0))
    return t0


t0 = gps_evento(EVENTO)
inicio = t0 - MEDIA_VENTANA
fin = t0 + MEDIA_VENTANA

In [ ]:
ventana = fin - inicio
print(ventana)
print(t0)

In [ ]:
def cargar(ifo):
    # Descarga de GWOSC solo la primera vez; después lee la copia de datos/
    archivo = DATOS / f"{ifo}_{EVENTO}_{2 * MEDIA_VENTANA}s.hdf5"
    if archivo.exists():
        return TimeSeries.read(archivo)
    serie = TimeSeries.fetch_open_data(ifo, inicio, fin)
    serie.write(archivo)
    return serie


datos_h1 = cargar("H1")
datos_l1 = cargar("L1")

In [ ]:
print(datos_h1)
print(datos_l1)

In [ ]:
datos_h1.plot();

In [ ]:
datos_l1.plot();

In [ ]:
asd_h1 = datos_h1.asd(fftlength=DURACION_FFT)
asd_l1 = datos_l1.asd(fftlength=DURACION_FFT)
print(asd_l1)

In [ ]:
plot = asd_h1.plot(label="H1")
ax = plot.gca()
ax.plot(asd_l1, label="L1")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(10, 2000)
ax.legend();

In [ ]:
def blanquear(serie):
    return serie.whiten(fftlength=DURACION_FFT).bandpass(*BANDA)


blanqueado_h1 = blanquear(datos_h1)
blanqueado_l1 = blanquear(datos_l1)
chirp_h1 = blanqueado_h1.crop(t0 + RECORTE[0], t0 + RECORTE[1])
chirp_l1 = blanqueado_l1.crop(t0 + RECORTE[0], t0 + RECORTE[1])

In [ ]:
plot = chirp_h1.plot(label="H1")
ax = plot.gca()
ax.plot(chirp_l1, label="L1")
ax.legend();

In [ ]:
def retraso_entre(a, b):
    # Los brazos de L1 están girados respecto a los de H1: la misma onda puede llegar
    # con el signo invertido, así que se busca el máximo de |correlación|.
    n = int(RETRASO_MAXIMO * a.sample_rate.value)
    x, y = a.value, b.value
    desplazamientos = np.arange(-n, n + 1)
    correlacion = np.array([np.dot(x[n:-n], np.roll(y, k)[n:-n]) for k in desplazamientos])
    correlacion /= np.sqrt(np.dot(x[n:-n], x[n:-n]) * np.dot(y[n:-n], y[n:-n]))
    mejor = np.argmax(np.abs(correlacion))
    return desplazamientos[mejor] / a.sample_rate.value, correlacion[mejor]


retraso, correlacion = retraso_entre(chirp_h1, chirp_l1)
print(f"L1 llega {retraso * 1000:.1f} ms antes que H1 (correlación {correlacion:+.2f})")

In [ ]:
alineado_l1 = np.sign(correlacion) * chirp_l1
alineado_l1.shift(retraso)

plot = chirp_h1.plot(label="H1")
ax = plot.gca()
ax.plot(alineado_l1, label="L1 desplazado e invertido")
ax.legend();

In [ ]:
def graficar_q(serie, ifo):
    # q_transform blanquea por su cuenta: recibe los datos crudos, no los blanqueados.
    q = serie.q_transform(outseg=(t0 + RECORTE[0], t0 + RECORTE[1]), frange=(20, 500), qrange=(4, 64))
    plot = q.plot()
    ax = plot.gca()
    ax.set_yscale("log")
    ax.set_title(ifo)
    ax.colorbar(label="Energía normalizada")
    return plot

In [ ]:
graficar_q(datos_h1, "H1");

In [ ]:
graficar_q(datos_l1, "L1");